# Credit Risk Decision Engine — Feature Engineering

Production and benchmark-only transformations are deliberately separate. Both are target-independent and imported from reusable modules.


In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.components.data_loader import load_training_data
from src.components.data_preprocessor import select_production_raw_features
from src.components.feature_contract import PRODUCTION_ENGINEERED_FEATURES, PRODUCTION_RAW_FEATURES
from src.components.feature_engineering import BENCHMARK_ONLY_ENGINEERED_FEATURES, create_benchmark_features, create_deployable_features


## Apply the deployable contract and transformation


In [2]:
df = load_training_data()
source_snapshot = df.copy(deep=True)
deployable_raw = select_production_raw_features(df)
deployable = create_deployable_features(deployable_raw)
pd.testing.assert_frame_equal(df, source_snapshot)
assert set(PRODUCTION_RAW_FEATURES).issubset(deployable.columns)
assert set(PRODUCTION_ENGINEERED_FEATURES).issubset(deployable.columns)
assert not set(BENCHMARK_ONLY_ENGINEERED_FEATURES).intersection(deployable.columns)
assert not np.isinf(deployable.select_dtypes(include=np.number).to_numpy()).any()
display(deployable.head())


,DAYS_BIRTH,DAYS_EMPLOYED,CNT_FAM_MEMBERS,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_CONTRACT_TYPE,NAME_INCOME_TYPE,...,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_ANNUITY_RATIO,CREDIT_GOODS_RATIO,INCOME_PER_PERSON,CREDIT_PER_PERSON,ANNUITY_PER_PERSON,CHILDREN_FAMILY_RATIO
0,-9461,-637,1.0,0,202500.0,406597.5,24700.5,351000.0,Cash loans,Working,...,1.744011,0.067329,2.007889,0.121978,16.461104,1.158397,202500.0,406597.50,24700.50,0.0
1,-16765,-1188,2.0,0,270000.0,1293502.5,35698.5,1129500.0,Cash loans,State servant,...,3.252567,0.070862,4.790750,0.132217,36.234085,1.145199,135000.0,646751.25,17849.25,0.0
2,-19046,-225,1.0,0,67500.0,135000.0,6750.0,135000.0,Revolving loans,Working,...,0.616016,0.011814,2.000000,0.100000,20.000000,1.000000,67500.0,135000.00,6750.00,0.0
3,-19005,-3039,2.0,0,135000.0,312682.5,29686.5,297000.0,Cash loans,Working,...,8.320329,0.159905,2.316167,0.219900,10.532818,1.052803,67500.0,156341.25,14843.25,0.0
4,-19932,-3038,1.0,0,121500.0,513000.0,21865.5,513000.0,Cash loans,Working,...,8.317591,0.152418,4.222222,0.179963,23.461618,1.000000,121500.0,513000.00,21865.50,0.0


Production engineering creates age/employment stability, affordability, repayment burden, and household-capacity features. Division by zero becomes missing, `DAYS_EMPLOYED = 365243` becomes a missing cleaned value plus an anomaly flag, and no feature uses `TARGET`.


## Benchmark-only features remain available for research


In [3]:
benchmark = create_benchmark_features(df)
assert set(BENCHMARK_ONLY_ENGINEERED_FEATURES).issubset(benchmark.columns)
display(benchmark[list(BENCHMARK_ONLY_ENGINEERED_FEATURES)].head())


,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_COUNT,DOCUMENT_COUNT,CONTACT_COUNT,DEF_30_SOCIAL_RATIO,DEF_60_SOCIAL_RATIO
0,0.161787,0.083037,0.262949,0.092026,3,1,4,1.0,1.0
1,0.466757,0.311267,0.622246,0.219895,2,1,4,0.0,0.0
2,0.642739,0.555912,0.729567,0.122792,2,0,5,NaN,NaN
3,0.650442,0.650442,0.650442,NaN,1,1,3,0.0,0.0
4,0.322738,0.322738,0.322738,NaN,1,1,3,NaN,NaN


External-score aggregates, document/contact counts, and social-circle ratios are not created during production inference. Keeping a separate benchmark function prevents accidental training-serving drift while preserving valid historical experiments.


## Standalone signal is descriptive, not model selection


In [4]:
def standalone_auc(frame, column):
    sample = pd.DataFrame({"feature": frame[column], "target": df["TARGET"]}).dropna()
    if sample["feature"].nunique() < 2:
        return np.nan
    auc = roc_auc_score(sample["target"], sample["feature"])
    return max(auc, 1 - auc)

signal = pd.Series({name: standalone_auc(deployable, name) for name in PRODUCTION_ENGINEERED_FEATURES})
display(signal.sort_values(ascending=False).rename("standalone_auc").to_frame())


,standalone_auc
AGE_YEARS,0.583003
EMPLOYMENT_YEARS,0.582391
DAYS_EMPLOYED_CLEAN,0.582391
EMPLOYMENT_AGE_RATIO,0.569696
CREDIT_GOODS_RATIO,0.569198
DAYS_EMPLOYED_ANOMALY,0.532432
CREDIT_ANNUITY_RATIO,0.532056
ANNUITY_INCOME_RATIO,0.519531
CREDIT_PER_PERSON,0.518761
CHILDREN_FAMILY_RATIO,0.517896
